# Expected goals: how good was that chance?
Every shot of the 2015/16 Premier League season (380 matches) from StatsBomb Open Data.

Run from the project folder:  `python src/get_data.py`  (once)  then  `python src/analysis.py`

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
FIG, RES = ROOT / "figures", ROOT / "results"
FIG.mkdir(exist_ok=True), RES.mkdir(exist_ok=True)
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8982"
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "savefig.bbox": "tight", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e4dd", "axes.axisbelow": True,
                     "font.size": 10.5, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})
funnel = []


def log(label, df, note):
    funnel.append({"label": label, "rows": int(len(df)), "note": note})
    print(f"{label:<45s} {len(df):>7,d} rows   {note}")

## 1. Load
`get_data.py` already turned 380 nested JSON files into one row per shot.

In [2]:
shots = pd.read_csv(ROOT / "data" / "shots.csv", parse_dates=["match_date"])
matches = pd.read_csv(ROOT / "data" / "matches.csv", parse_dates=["match_date"])
log("All shots, Premier League 2015/16", shots, f"{shots.match_id.nunique()} matches")
print(shots.isna().mean().round(3)[lambda s: s > 0])

All shots, Premier League 2015/16               9,908 rows   380 matches
first_time           0.703
one_on_one           0.953
open_goal            0.989
aerial_won           0.908
follows_dribble      0.999
under_pressure       0.784
defenders_in_cone    0.007
gk_in_cone           0.007
gk_dist_to_goal      0.009
nearest_defender     0.009
dtype: float64


## 2. Clean
### 2.1 Six columns are 85 to 99% empty, and that is not a problem
StatsBomb writes flags such as `first_time` or `under_pressure` only when they are true. An empty cell
means **False**, by design. Here `fillna(False)` is the correct reading of the documentation. Using
`dropna()` would keep only the handful of shots where every flag happened to be true.

In [3]:
flags = ["first_time", "one_on_one", "open_goal", "aerial_won", "follows_dribble", "under_pressure"]
print("Share empty before:", shots[flags].isna().mean().round(3).to_dict())
shots[flags] = shots[flags].astype("boolean").fillna(False).astype(int)
shots["goal"] = (shots["outcome"] == "Goal").astype(int)

Share empty before: {'first_time': 0.703, 'one_on_one': 0.953, 'open_goal': 0.989, 'aerial_won': 0.908, 'follows_dribble': 0.999, 'under_pressure': 0.784}


### 2.2 Penalties are a different event
A penalty is always taken from the same spot with nobody in the way. Leaving penalties in would teach the
model that "12 yards out, central" is a 75% chance, which is false for open play. They are reported
separately and removed from the model, as published expected-goals models do.

In [4]:
pens = shots[shots["shot_type"] == "Penalty"]
print(f"Penalties: {len(pens)} taken, {pens.goal.sum()} scored ({pens.goal.mean():.1%})")
df = shots[shots["shot_type"] != "Penalty"].copy()
log("Non-penalty shots", df, f"{len(pens)} penalties ({pens.goal.mean():.0%} scored) set aside")

Penalties: 91 taken, 74 scored (81.3%)
Non-penalty shots                               9,817 rows   91 penalties (81% scored) set aside


### 2.3 Turn pitch coordinates into football
Raw `x, y` are positions on a 120 x 80 yard pitch. A model can use them, but two engineered features carry
the meaning directly: **distance** to the centre of the goal, and the **angle** the goal mouth covers as
seen from the ball (wide in front of goal, a sliver from the byline).

In [5]:
gx, post_l, post_r = 120.0, 36.0, 44.0
df["distance"] = np.hypot(gx - df["x"], 40 - df["y"])
a = np.arctan2(post_r - df["y"], gx - df["x"]) - np.arctan2(post_l - df["y"], gx - df["x"])
df["angle"] = np.degrees(np.abs(a))
print("Shots with no freeze frame (player positions unknown):", int((~df["has_freeze_frame"]).sum()))
for c in ["defenders_in_cone", "gk_in_cone"]:
    df[c] = pd.to_numeric(df[c].replace({"True": 1, "False": 0, True: 1, False: 0}), errors="coerce")
print(f"Goal rate, non-penalty shots: {df.goal.mean():.3f}  ({df.goal.sum()} goals from {len(df):,} shots)")

Shots with no freeze frame (player positions unknown): 0
Goal rate, non-penalty shots: 0.093  (914 goals from 9,817 shots)


## 3. Explore

In [6]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
miss, hit = df[df.goal == 0], df[df.goal == 1]
ax.scatter(miss["y"], miss["x"], s=7, color=BLUE, alpha=0.18, lw=0, label=f"no goal ({len(miss):,})")
ax.scatter(hit["y"], hit["x"], s=14, color=ORANGE, alpha=0.85, lw=0, label=f"goal ({len(hit):,})")
for (x0, x1, y0, y1) in [(18, 62, 102, 120), (30, 50, 114, 120)]:               # penalty area and six-yard box
    ax.plot([x0, x0, x1, x1], [y1, y0, y0, y1], color="#52514e", lw=1.2)
ax.plot([36, 44], [120, 120], color="#0b0b0b", lw=4)
ax.set(xlim=(0, 80), ylim=(78, 121), xticks=[], yticks=[], title="9,800 shots: goals come from a small zone in front of goal")
ax.grid(False); ax.spines[["left", "bottom"]].set_visible(False); ax.legend(frameon=False, loc="lower left", markerscale=2)
ax.set_aspect("equal")
fig.savefig(FIG / "fig1_shot_map.png"); plt.close(fig)

In [7]:
dist_bins = [0, 6, 10, 14, 18, 22, 26, 30, 100]
by_dist = df.groupby(pd.cut(df["distance"], dist_bins), observed=True)["goal"].agg(["mean", "size"])
labels = ["<6", "6-10", "10-14", "14-18", "18-22", "22-26", "26-30", "30+"]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(labels, by_dist["mean"] * 100, color=BLUE, width=0.72)
for x, (m, n) in enumerate(zip(by_dist["mean"], by_dist["size"])):
    ax.text(x, m * 100 + 1, f"{m:.0%}\nn={n:,}", ha="center", fontsize=8.5, color="#52514e")
ax.set(xlabel="Distance from goal (yards)", ylabel="% of shots scored", ylim=(0, by_dist["mean"].max() * 122), title="Conversion collapses with distance")
fig.savefig(FIG / "fig2_distance.png"); plt.close(fig)

In [8]:
close = df[df["distance"].between(8, 16)]
by_block = close.groupby(close["defenders_in_cone"].clip(upper=3))["goal"].agg(["mean", "size"])
by_body = close.groupby(close["body_part"].replace({"Right Foot": "Foot", "Left Foot": "Foot"}))["goal"].agg(["mean", "size"]).loc[["Foot", "Head"]]
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4), sharey=True)
axes[0].bar(["0", "1", "2", "3+"], by_block["mean"] * 100, color=BLUE, width=0.7)
axes[0].set(xlabel="Defenders between ball and goal", ylabel="% scored", title="Shots from 8 to 16 yards")
axes[1].bar(by_body.index, by_body["mean"] * 100, color=BLUE, width=0.5)
axes[1].set(xlabel="Body part", title=" ")
for ax, t in zip(axes, [by_block, by_body]):
    for x, (m, n) in enumerate(zip(t["mean"], t["size"])):
        ax.text(x, m * 100 + 0.8, f"{m:.0%}\nn={n:,}", ha="center", fontsize=8.5, color="#52514e")
axes[0].set_ylim(0, by_block["mean"].max() * 125)
fig.savefig(FIG / "fig3_context.png"); plt.close(fig)

## 4. Split by date
Train on the first 80% of matches, test on the final 20% (the run-in). All shots from one match stay on the
same side of the split, because shots within a match are not independent.

In [9]:
order = matches.sort_values("match_date")["match_id"].tolist()
cut = int(len(order) * 0.8)
train, test = df[df["match_id"].isin(order[:cut])], df[df["match_id"].isin(order[cut:])]
y_train, y_test = train["goal"], test["goal"]
cutoff_date = matches.sort_values("match_date")["match_date"].iloc[cut].date()
print(f"train {len(train):,} shots from {cut} matches ({y_train.mean():.1%} goals) | test {len(test):,} shots from {len(order) - cut} matches from {cutoff_date} ({y_test.mean():.1%} goals)")

train 7,787 shots from 304 matches (9.1% goals) | test 2,030 shots from 76 matches from 2016-04-02 (10.2% goals)


## 5. Models
The standard metric for expected goals is not accuracy (predicting "no goal" every time is 90% accurate and useless).
It is **log loss** and **Brier score**: how good are the probabilities? Lower is better. StatsBomb's own model,
which uses more information than is public (shot height, exact goalkeeper stance), is the professional benchmark.

In [10]:
geo = ["distance", "angle"]
num = geo + ["defenders_in_cone", "gk_in_cone", "gk_dist_to_goal", "nearest_defender"] + flags
cat = ["body_part", "shot_type", "technique", "play_pattern"]


def logit(n, c):
    steps = [("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), n)]
    if c:
        steps.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20), c))
    return make_pipeline(ColumnTransformer(steps), LogisticRegression(max_iter=1000))


def gbm(n, c):
    pre = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), c), ("num", "passthrough", n)])
    return make_pipeline(pre, HistGradientBoostingClassifier(categorical_features=list(range(len(c))), max_iter=300, learning_rate=0.04, max_depth=4,
                                                             l2_regularization=1.0, early_stopping=True, validation_fraction=0.2, random_state=42))


def score(name, p, y):
    return {"name": name, "logloss": log_loss(y, p), "brier": brier_score_loss(y, p), "auc": roc_auc_score(y, p)}


results = [score("Baseline: every shot gets the average rate", np.full(len(test), y_train.mean()), y_test)]
fitted = {}
for name, make, n, c in [("Logistic regression: distance + angle", logit, geo, []), ("Logistic regression: all features", logit, num, cat), ("Gradient boosting: all features", gbm, num, cat)]:
    m = make(n, c).fit(train[c + n], y_train)
    p = m.predict_proba(test[c + n])[:, 1]
    results.append(score(name, p, y_test)); fitted[name] = (m, c + n, p)
results.append(score("Benchmark: StatsBomb's own xG", test["statsbomb_xg"].clip(1e-6, 1 - 1e-6), y_test))
print(pd.DataFrame(results).round(4).to_string())

                                         name  logloss   brier     auc
0  Baseline: every shot gets the average rate   0.3301  0.0917  0.5000
1       Logistic regression: distance + angle   0.3006  0.0861  0.7270
2           Logistic regression: all features   0.2827  0.0807  0.7684
3             Gradient boosting: all features   0.2847  0.0809  0.7582
4               Benchmark: StatsBomb's own xG   0.2725  0.0770  0.7859


In [11]:
r = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.barh(r["name"][::-1], r["logloss"][::-1], color=[AQUA, BLUE, BLUE, BLUE, GREY], height=0.65)
for y, v in enumerate(r["logloss"][::-1]):
    ax.text(v + 0.002, y, f"{v:.4f}", va="center", fontsize=9)
ax.set(xlim=(0.24, r["logloss"].max() * 1.06), xlabel="Log loss on the final 20% of matches (lower is better)", title="Distance and angle do most of the work; context closes most of the gap to the professional model")
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "fig4_models.png"); plt.close(fig)

In [12]:
best_name = min(fitted, key=lambda k: next(x["logloss"] for x in results if x["name"] == k))
best_m, best_cols, best_p = fitted[best_name]
print("Best own model:", best_name)
cal = pd.DataFrame({"p": best_p, "y": y_test.values}); cal["bin"] = pd.qcut(cal["p"], 8, duplicates="drop")
calib = cal.groupby("bin", observed=True).agg(pred=("p", "mean"), actual=("y", "mean"), n=("y", "size"))
print(calib.round(3))
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, .6], [0, .6], color=GREY, ls="--", lw=1); ax.plot(calib["pred"], calib["actual"], marker="o", color=ORANGE, lw=2)
ax.set(xlabel="Predicted chance of a goal (xG)", ylabel="Share actually scored", title="Calibration (eight equal-sized groups)")
fig.savefig(FIG / "fig5_calibration.png"); plt.close(fig)
pi = permutation_importance(best_m, test[best_cols], y_test, scoring="neg_log_loss", n_repeats=10, random_state=1)
nice = {"distance": "Distance to goal", "angle": "Angle of the goal mouth", "defenders_in_cone": "Defenders in the way", "gk_in_cone": "Goalkeeper in the way", "gk_dist_to_goal": "Goalkeeper off the line",
        "nearest_defender": "Distance to nearest defender", "first_time": "First-time shot", "one_on_one": "One-on-one", "open_goal": "Open goal", "aerial_won": "Aerial duel won",
        "follows_dribble": "After a dribble", "under_pressure": "Under pressure", "body_part": "Body part", "shot_type": "Open play / free kick / corner", "technique": "Technique (volley, lob…)", "play_pattern": "Phase of play"}
imp = pd.Series(pi.importances_mean, index=best_cols).sort_values(ascending=False)
print(imp.round(4))

Best own model: Logistic regression: all features
                   pred  actual    n
bin                                 
(0.0009, 0.02]    0.014   0.016  254
(0.02, 0.0315]    0.026   0.028  254
(0.0315, 0.042]   0.037   0.040  253
(0.042, 0.0548]   0.049   0.051  254
(0.0548, 0.0712]  0.063   0.083  254
(0.0712, 0.103]   0.085   0.126  253
(0.103, 0.177]    0.133   0.146  254
(0.177, 0.968]    0.329   0.327  254


distance             0.0453
angle                0.0157
nearest_defender     0.0074
body_part            0.0059
defenders_in_cone    0.0031
gk_dist_to_goal      0.0030
shot_type            0.0029
aerial_won           0.0022
technique            0.0018
play_pattern         0.0006
first_time           0.0003
one_on_one           0.0000
follows_dribble      0.0000
gk_in_cone          -0.0000
under_pressure      -0.0000
open_goal           -0.0002
dtype: float64


## 6. The decision: who finished above their chances, and should a club pay for it?
For season totals every shot needs a prediction from a model that did **not** train on that shot's match.
`cross_val_predict` with `GroupKFold` (groups = matches) does exactly that.

In [13]:
make_best = {"Logistic regression: distance + angle": lambda: logit(geo, []), "Logistic regression: all features": lambda: logit(num, cat), "Gradient boosting: all features": lambda: gbm(num, cat)}[best_name]
df["xg"] = cross_val_predict(make_best(), df[best_cols], df["goal"], groups=df["match_id"], cv=GroupKFold(5), method="predict_proba")[:, 1]
print(f"Season check: goals {df.goal.sum()}, our xG {df.xg.sum():.1f}, StatsBomb xG {df.statsbomb_xg.sum():.1f}")
players = (df.groupby(["player", "team"]).agg(shots=("goal", "size"), goals=("goal", "sum"), xg=("xg", "sum"), sb_xg=("statsbomb_xg", "sum")).reset_index())
players["diff"] = players["goals"] - players["xg"]
top = players.sort_values("goals", ascending=False).head(12)
print(top.round(2).to_string(index=False))
teams = df.groupby("team").agg(shots=("goal", "size"), goals=("goal", "sum"), xg=("xg", "sum")).assign(diff=lambda t: t.goals - t.xg).sort_values("diff", ascending=False)
print(teams.round(1).to_string())

fig, ax = plt.subplots(figsize=(7.5, 4.4))
short = {"Sergio Leonel Agüero del Castillo": "Agüero", "André Ayew Pelé": "A. Ayew", "Romelu Lukaku Menama": "Lukaku", "Alexis Alejandro Sánchez Sánchez": "Alexis Sánchez", "Diego da Silva Costa": "Diego Costa", "Odion Jude Ighalo": "Ighalo"}
t = top.sort_values("diff")
ax.barh(t["player"].map(lambda p: short.get(p, p.split()[-1])) + " (" + t["goals"].astype(str) + ")", t["diff"], color=[ORANGE if v > 0 else BLUE for v in t["diff"]], height=0.7)
ax.axvline(0, color="#0b0b0b", lw=1)
ax.set(xlabel="Non-penalty goals minus expected goals (positive = scored more than the chances were worth)", title="Top 12 scorers 2015/16 (non-penalty goals in brackets)")
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "fig6_players.png"); plt.close(fig)

# Does finishing "above xG" persist? Split each player's season in half by date and compare.
half = df["match_id"].isin(order[: len(order) // 2])
h1 = df[half].groupby("player").agg(s1=("goal", "size"), d1=("goal", "sum"), x1=("xg", "sum"))
h2 = df[~half].groupby("player").agg(s2=("goal", "size"), d2=("goal", "sum"), x2=("xg", "sum"))
both = h1.join(h2, how="inner").query("s1 >= 20 and s2 >= 20")
both["over1"], both["over2"] = (both.d1 - both.x1) / both.s1, (both.d2 - both.x2) / both.s2
both["conv1"], both["conv2"], both["xgps1"], both["xgps2"] = both.d1 / both.s1, both.d2 / both.s2, both.x1 / both.s1, both.x2 / both.s2
persist = {"n_players": int(len(both)), "r_overperformance": float(both["over1"].corr(both["over2"])), "r_xg_per_shot": float(both["xgps1"].corr(both["xgps2"])), "r_conversion": float(both["conv1"].corr(both["conv2"]))}
print(persist)

Season check: goals 914, our xG 916.2, StatsBomb xG 900.1
                           player              team  shots  goals    xg  sb_xg  diff
                       Harry Kane Tottenham Hotspur    153     20 17.11  17.72  2.89
Sergio Leonel Agüero del Castillo   Manchester City    113     20 14.63  13.76  5.37
                      Jamie Vardy    Leicester City    112     19 19.50  17.60 -0.50
             Romelu Lukaku Menama           Everton    114     17 16.67  16.59  0.33
                   Olivier Giroud           Arsenal    102     15 13.25  12.23  1.75
                Odion Jude Ighalo           Watford    111     15 18.00  16.92 -3.00
                    Jermain Defoe        Sunderland     87     14 12.51  12.43  1.49
                     Riyad Mahrez    Leicester City     81     13  7.26   7.46  5.74
 Alexis Alejandro Sánchez Sánchez           Arsenal    111     13 11.35  11.26  1.65
                  André Ayew Pelé      Swansea City     69     12  7.14   7.59  4.86
       

In [14]:
edges = np.linspace(0, 1, 51)
posh, _ = np.histogram(best_p[y_test.values == 1], bins=edges); negh, _ = np.histogram(best_p[y_test.values == 0], bins=edges)
out = {"n_raw": int(len(shots)), "n_matches": int(shots.match_id.nunique()), "n_model": int(len(df)), "goal_rate": float(df.goal.mean()), "goals": int(df.goal.sum()),
       "penalties": {"n": int(len(pens)), "scored": int(pens.goal.sum())}, "flags_empty": shots[flags].eq(0).mean().round(4).to_dict(), "no_freeze_frame": int((~df["has_freeze_frame"]).sum()),
       "train": {"n": int(len(train)), "rate": float(y_train.mean()), "matches": cut}, "test": {"n": int(len(test)), "rate": float(y_test.mean()), "matches": len(order) - cut, "from": str(cutoff_date)},
       "models": results, "best": best_name, "funnel": funnel, "season": {"goals": int(df.goal.sum()), "xg": float(df.xg.sum()), "sb_xg": float(df.statsbomb_xg.sum())},
       "by_dist": [{"bin": l, "rate": float(m), "n": int(n)} for l, (m, n) in zip(labels, zip(by_dist["mean"], by_dist["size"]))],
       "by_block": [{"bin": str(int(k)) if k < 3 else "3+", "rate": float(v["mean"]), "n": int(v["size"])} for k, v in by_block.iterrows()],
       "by_body": [{"bin": k, "rate": float(v["mean"]), "n": int(v["size"])} for k, v in by_body.iterrows()],
       "calibration": [{"pred": float(x.pred), "actual": float(x.actual), "n": int(x.n)} for x in calib.itertuples()],
       "importance": [{"name": nice[k], "value": float(v)} for k, v in imp.items()],
       "players": top.round(3).to_dict("records"), "teams": teams.round(2).reset_index().to_dict("records"), "persist": persist,
       "bins": {"pos": posh.tolist(), "neg": negh.tolist()}}
(RES / "metrics.json").write_text(json.dumps(out, indent=1))
print("Saved figures/ and results/metrics.json")

Saved figures/ and results/metrics.json
